#### Initialize

In [17]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
DF_RANKED = pd.read_csv(RANKED_BUCKET_PATH / "places_scored_level_1.csv")

#### Representation

In [39]:
df_ranked = DF_RANKED.copy()

# FIND reasonably represented cuisine types
df_ranked['cuisineType'] = df_ranked['cuisineType'].fillna('Unspecified')
df_reasonabily_represented_types = df_ranked['cuisineType'].value_counts() \
    / len(df_ranked[df_ranked['cuisineType']!='Unspecified']) >= 0.02

# GET normalized score per cuisine type
df_ranked['represented'] = df_ranked['cuisineType'].map(df_reasonabily_represented_types)
for score_idx in [0, 1, 2]:
    df_ranked.loc[~df_ranked['represented'], f"cnormal_{score_idx}"] = (
        df_ranked.loc[~df_ranked['represented'], [f"wilson_{score_idx}"]].rank(pct=True)
    )
    df_ranked.loc[df_ranked['represented'], f"cnormal_{score_idx}"] = (
        df_ranked.groupby('cuisineType')[f"wilson_{score_idx}"].rank(pct=True)
    )

#### Export

In [40]:
df_ranked.to_csv(RANKED_BUCKET_PATH / "places_ranked_level_2.csv", index=False)